In [137]:
# Mount to Google Drive & Switch to the dataset directory
import os
# Local directory switch removed; use run.py from the project root.
os.listdir()
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [138]:
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision import transforms
from PIL import Image
import numpy as np


class CustomDataset(Dataset):
    def __init__(self, npz_path):
        npz_data = np.load(npz_path)
        self.images = npz_data["images"] # (N, 3, 128, 128) in np.uint8
        self.labels = npz_data["labels"] # (N,) in np.int64
        assert self.images.shape[0] == self.labels.shape[0]
        print(f"{npz_path}: images shape {self.images.shape}, "
              f"labels shape {self.labels.shape}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx]) / 255 # convert to [0, 1] range
        label = torch.tensor(self.labels[idx])
        return image, label

train_dataset = CustomDataset("train.npz")

def get_dataloaders(dataset, val_ratio=0.1, batch_size=128):
    from torch.utils.data import DataLoader, random_split

    val_size = int(len(dataset) * val_ratio)
    train_size = len(dataset) - val_size
    train_set, val_set = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader

train_loader, val_loader = get_dataloaders(train_dataset)



train.npz: images shape (18900, 3, 128, 128), labels shape (18900,)


In [139]:
# sample data batch
images, labels = next(iter(train_loader))
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")

images shape: torch.Size([128, 3, 128, 128])
labels shape: torch.Size([128])


In [140]:
import matplotlib.pyplot as plt

# load label text
with open('label2type.txt', 'r') as f:
    label2type = eval(f.read())

# plot samples
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
for i, ax in enumerate(axes):
    label = labels[i].item()
    # (C, H, W) to (H, W, C) for plotting
    ax.imshow(images[i].numpy().transpose(1, 2, 0))
    ax.axis('off')
    ax.set_title(f"{label}: {label2type[label]}")
plt.tight_layout()
plt.show()

<Figure size 1200x400 with 4 Axes>

In [141]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18

# 获取标签类别数量（确保定义了 train_dataset）
num_classes = len(set(train_dataset.labels))

class ConvVAE(nn.Module):
    def __init__(self, input_channels=3, latent_channels=64):
        super().__init__()

        # --- Encoder: 使用 ResNet18 的前 3 个 block ---
        resnet = resnet18(weights='DEFAULT')
        self.encoder = nn.Sequential(
            resnet.conv1,    # [B, 64, 64, 64]
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,  # [B, 64, 32, 32]
            resnet.layer1,   # [B, 64, 32, 32]
            resnet.layer2,   # [B, 128, 16, 16]
            resnet.layer3    # [B, 256, 8, 8]
        )

        # --- Bottleneck ---
        self.quant_conv = nn.Conv2d(256, latent_channels * 2, kernel_size=1)    # 输出 mean/logvar
        self.post_quant_conv = nn.Conv2d(latent_channels, 256, kernel_size=1)  # 输入 latent z

        # --- Decoder: 三层反卷积，还原到 128×128 ---
        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),           # [B,128,8,8]
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),   # [B,64,16,16]
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),    # [B,32,32,32]
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1),    # [B,16,64,64]
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(16, input_channels, kernel_size=4, stride=2, padding=1),  # [B,3,128,128]
            nn.Sigmoid()
        )

        # --- Classifier: 更强的分类器结构 ---
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(latent_channels, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def preprocess(self, x):
        return 2 * x - 1

    def encode(self, x):
        x = self.preprocess(x)
        h = self.encoder(x)               # [B, 256, 8, 8]
        h = self.quant_conv(h)           # [B, 2*latent, 8, 8]
        mean, logvar = torch.chunk(h, 2, dim=1)
        if self.training:
            logvar = logvar.clamp(-20.0, 10.0)
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z = mean + eps * std
        else:
            z = mean
        return z, mean, logvar

    def decode(self, z):
        h = self.post_quant_conv(z)
        return self.decoder(h)

    def forward(self, x):
        z, mean, logvar = self.encode(x)
        x_recon = self.decode(z)
        logits = self.classifier(z)
        return x_recon, z, mean, logvar, logits


In [ ]:
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm


def combined_vae_loss(x, x_recon, mean, logvar, logits, labels, kl_weight=1e-4, ce_weight=0.1):
    recon_loss = F.mse_loss(x_recon, x)
    kl_loss = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
    ce_loss = F.cross_entropy(logits, labels, label_smoothing=0.1)
    total_loss = recon_loss + kl_weight * kl_loss + ce_weight * ce_loss
    return total_loss, recon_loss, kl_loss, ce_loss

def freeze_decoder(model):
    for param in model.decoder.parameters():
        param.requires_grad = False
    for param in model.post_quant_conv.parameters():
        param.requires_grad = False

def unfreeze_decoder(model):
    for param in model.decoder.parameters():
        param.requires_grad = True
    for param in model.post_quant_conv.parameters():
        param.requires_grad = True

def evaluate_mse(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_recon, *_ = model(x)
            loss = F.mse_loss(x_recon, x, reduction='sum')
            total_loss += loss.item()
    mse = total_loss / len(dataloader.dataset)
    return mse

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def extract_latent_and_labels(model, dataloader, device, use_mean=True):
    model.eval()
    latents, labels = [], []
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            _, z, mean, _, _ = model(x)
            latent = mean if use_mean else z
            latents.append(latent.view(latent.size(0), -1).cpu())
            labels.append(y)
    return torch.cat(latents), torch.cat(labels)

def evaluate_linear_probing(model, dataloader, device, use_mean=True):
    X, y = extract_latent_and_labels(model, dataloader, device, use_mean)
    X = X.numpy()
    y = y.numpy()
    split = int(0.8 * len(X))
    clf = LogisticRegression(max_iter=2000, solver='lbfgs', multi_class='multinomial')
    clf.fit(X[:split], y[:split])
    acc = accuracy_score(y[split:], clf.predict(X[split:]))
    print(f"Linear probing accuracy: {acc:.4f}")
    return acc




# ----- Training -----
def train_vae(model, dataloader, optimizer, device, num_epochs=1, warmup_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        if epoch < warmup_epochs:
            freeze_decoder(model)
            ce_weight = 2.0  # 强化分类任务
        else:
            unfreeze_decoder(model)
            ce_weight = 1.0

        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        # loop = dataloader
        for images, labels in loop:
            x, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            x_recon, z, mean, logvar, logits = model(x)
            loss, recon_loss, kl_loss, ce_loss = combined_vae_loss(x, x_recon, mean, logvar, logits, labels,kl_weight=1e-4, ce_weight=ce_weight)

            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item(), recon=recon_loss.item(), kl=kl_loss.item(), ce=ce_loss.item())

                # ✅ 每个 epoch 后评估 Validation MSE 和 LP accuracy
        val_mse = evaluate_mse(model, val_loader, device)
        val_acc = evaluate_linear_probing(model, val_loader, device, use_mean=True)
        print(f"[Validation] MSE: {val_mse:.2f}, LP Accuracy: {val_acc:.4f}")




# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model
model = ConvVAE(input_channels=3, latent_channels=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)


# Train
train_vae(model, train_loader, optimizer, device, num_epochs=20)

Epoch 1/20: 100%|██████████| 133/133 [05:14<00:00,  2.37s/it, ce=4.74, kl=0.53, loss=9.69, recon=0.219] 


In [ ]:
# ----- Visualization -----
def plot_reconstructions(model, dataloader, device, num_images=8):
    model.eval()
    with torch.no_grad():
        x = next(iter(dataloader))[0].to(device)
        x_recon, z, *_ = model(x)
        x = x.cpu().numpy()
        x_recon = x_recon.cpu().numpy()
        print(f"Latent bottleneck dimension: {z.flatten(start_dim=1).shape[1]}")

        plt.figure(figsize=(16, 4))
        for i in range(num_images):
            # Original
            plt.subplot(2, num_images, i+1)
            plt.imshow(x[i].transpose(1, 2, 0))  # (C, H, W) -> (H, W, C)
            plt.axis('off')

            # Reconstruction
            plt.subplot(2, num_images, i+1+num_images)
            plt.imshow(x_recon[i].transpose(1, 2, 0))
            plt.axis('off')

        plt.show()

plot_reconstructions(model, train_loader, device, num_images=8)

Latent bottleneck dimension: 4096


<Figure size 1600x400 with 16 Axes>

In [ ]:
def evaluate_mse(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, _ in dataloader:
            x = x.to(device)
            x_recon, z, mean, logvar, logits = model(x)  # ✅ 完整解包
            loss = F.mse_loss(x_recon, x, reduction='sum')
            total_loss += loss.item()

    mse = total_loss / len(dataloader.dataset)
    return mse



In [ ]:
probe_classifier = nn.Sequential(
    nn.Linear(8 * 32 * 32, 256),
    nn.ReLU(),
    nn.Linear(256, 256),
    nn.ReLU(),
    nn.Linear(256, num_classes)
).to(device)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score
def extract_latents_and_labels(model, dataloader, device, use_mean=True):
    model.eval()
    latents = []
    labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            _, z, mean, _, _ = model(x)  # ✅ 根据你的模型结构解包 5 个返回值
            z_used = mean if use_mean else z
            z_flat = z_used.view(z_used.size(0), -1).cpu()
            latents.append(z_flat)
            labels.append(y)

    X = torch.cat(latents, dim=0).numpy()
    y = torch.cat(labels, dim=0).numpy()
    return X, y

# 提取 latent 特征
X_train, y_train = extract_latents_and_labels(model, train_loader, device, use_mean=True)
X_val, y_val = extract_latents_and_labels(model, val_loader, device, use_mean=True)

print(f"Original latent dim: {X_train.shape[1]}")

# ✅ PCA 降维
pca = PCA(n_components=256)  # 可调为 128、256、512
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

print(f"PCA reduced dim: {X_train_pca.shape[1]}")

# ✅ Logistic Regression
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_pca, y_train)

# ✅ 评估准确率
y_pred = clf.predict(X_val_pca)
acc = accuracy_score(y_val, y_pred)
print(f"PCA + Logistic Regression Accuracy: {acc:.4f}")



Original latent dim: 4096
PCA reduced dim: 256
PCA + Logistic Regression Accuracy: 0.0995


In [ ]:
# 运行 MSE 评估
val_mse = evaluate_mse(model, val_loader, device)
print(f"Validation MSE: {val_mse:.6f}")



Validation MSE: 805.862802


In [ ]:
# Submission

# 1) Save model weights
torch.save(model.state_dict(), "checkpoint.pt")

# 2) Prepare the 'Model' class for submission
with open("model.py", "r") as f:
    print(f.read())

# 3) Submit the model code & weights online

import torch
import torch.nn as nn
import torch.nn.functional as F

class Model(nn.Module):
    def __init__(self, input_channels=3, latent_channels=4):
        super().__init__()

        # --- Encoder ---
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=3, stride=2, padding=1),
            nn.ELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.ELU(),
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
        )

        # --- Bottleneck ---
        self.quant_conv = nn.Conv2d(256, latent_channels * 2, kernel_size=1)  # mean + logvar

        # --- Post-bottleneck projection ---
        self.post_quant_conv = nn.Conv2d(latent_channels, 256, kernel_size=1)

        # --- Decoder ---
        self.decoder = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stri

In [ ]:
# Metric 1: Recon MSE on test set (on value range [0, 1])
# Metric 2: Classification accuracy (linear probing with test set latents, 170 classes)
# Final Score: recon_mse / probing_accuracy (the lower the better)